In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [25]:
# Web search tool and web search schema are already built-in in Claude, so we do not need to write them

# Small schema stub that relates to the model we are using. It is going to expand into a much bigger schema behing the sceness
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
    "allowed_domains": ["nih.gov"]
}

In [28]:
messages = []
add_user_message(
    messages,
    """
    Cause of schizophrenia among children in the US. Limit to 3 sentences.
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_01B3d42VqYHa6t7vvohZhTXw', container=None, content=[ServerToolUseBlock(id='srvtoolu_01R4hE6yNPMaRmnMB1KMNHx5', caller=None, input={'query': 'childhood schizophrenia causes US'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EuAQCioIEBgCIiRiYzAzM2M3MS02ZWMyLTRlODgtOTE0OS1hMTUzNmQzMDc5Y2ISDJ2W+qIjyPJhy0z7RRoMWvq6wrYN4iVIggtIIjBxMNH5iBJ36aI9a2TOvTlh8Nm+JG0KB2sU+FZOUlxjYCUoyr8iBzbI66C3DdKOtlMq4w8FkOB8mYVT9K6yMnwhMvgkIWpKpiwbQu50eN/uT5N0kkd8aDilSha5+1P92KzVadR0ijCs+8Ah2eP5RXIIU31jHPl413mEQJzn5d2EFOkFPAHStyZr49a9kdfdlsY/pJM8o9RgvZyiz9UCCKbkTTP6rA/+8zesh51Krcrz3zNNHErOloHfdGKBTzKe7iIZPzY4+Lfwlf9OchE6y/naTmpcO8Wcp4AKQ0RA+HH2uZHOIGNACYy+CPcsP61E4ZwdjwkbBNfMS0lB6eRtq5MYKTyvGwWhV0UWVFc4JsHQFDumbn/Xy36o9K6fXWWHl87DTp+IguLI/zSq0+JOJtX6JozQ+kUvd1J9kqOEIcD3uF5NVDLtLaaFFbXGHs2FN7WrBWPsmgJy2NIqq+1A1aaWCmwiMIlY71FWWLaaa9FhRgcea094UukO4gI/SCxGeuXh4cX9wCf6roTa5+J8GA/3Ade1oCfk7thifPxq

In [29]:
# Render the response into a locally hosted human-readable webpage
import html
import threading
import webbrowser
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path


def render_response_to_html(response):
    answer_parts = []
    search_queries = []
    search_results = []
    citation_idx = 0

    for block in response.content:
        btype = getattr(block, "type", None)
        if btype == "text":
            text_html = html.escape(block.text).replace("\n", "<br>")
            cites = getattr(block, "citations", None) or []
            cite_html = ""
            for c in cites:
                citation_idx += 1
                url = getattr(c, "url", "") or ""
                title = getattr(c, "title", "") or url
                cite_html += (
                    f'<a class="citation" href="{html.escape(url)}" '
                    f'title="{html.escape(title)}" target="_blank">[{citation_idx}]</a>'
                )
            answer_parts.append(text_html + cite_html)
        elif btype == "server_tool_use":
            q = (block.input or {}).get("query", "")
            search_queries.append(q)
        elif btype == "web_search_tool_result":
            for item in block.content or []:
                search_results.append({
                    "title": getattr(item, "title", "") or "",
                    "url": getattr(item, "url", "") or "",
                })

    parts = ["""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Claude Web Search Result</title>
<style>
  body { font-family: -apple-system, system-ui, sans-serif; max-width: 760px;
         margin: 2rem auto; padding: 0 1rem; color: #222; line-height: 1.6; }
  h1 { font-size: 1.4rem; border-bottom: 1px solid #ddd; padding-bottom: .5rem; }
  h2 { font-size: 1.1rem; margin-top: 2rem; }
  .answer { background: #f7f7f8; padding: 1rem 1.25rem; border-radius: 8px; }
  .citation { font-size: .75rem; vertical-align: super; color: #2563eb;
              text-decoration: none; margin-left: 2px; }
  .results { list-style: none; padding: 0; }
  .results li { border: 1px solid #eee; border-radius: 8px;
                padding: .75rem 1rem; margin-bottom: .75rem; }
  .results a { color: #2563eb; text-decoration: none; font-weight: 600; }
  .results .url { color: #666; font-size: .8rem; word-break: break-all; }
  .query { color: #555; font-style: italic; }
</style>
</head>
<body>
<h1>Claude Web Search Result</h1>
"""]
    parts.append('<div class="answer">' + "".join(answer_parts) + "</div>")

    if search_queries:
        parts.append("<h2>Search queries</h2><ul>")
        for q in search_queries:
            parts.append(f'<li class="query">{html.escape(q)}</li>')
        parts.append("</ul>")

    if search_results:
        parts.append('<h2>Sources</h2><ul class="results">')
        for r in search_results:
            parts.append(
                f'<li><a href="{html.escape(r["url"])}" target="_blank">'
                f'{html.escape(r["title"]) or "(no title)"}</a>'
                f'<div class="url">{html.escape(r["url"])}</div></li>'
            )
        parts.append("</ul>")

    parts.append("</body></html>")
    return "".join(parts)


_server = None

def serve_html(html_text, port=8765):
    global _server
    out_dir = Path("./_web_search_output").resolve()
    out_dir.mkdir(exist_ok=True)
    (out_dir / "index.html").write_text(html_text, encoding="utf-8")

    class Handler(SimpleHTTPRequestHandler):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, directory=str(out_dir), **kwargs)
        def log_message(self, *a, **kw):
            pass

    if _server is not None:
        _server.shutdown()
        _server.server_close()

    _server = HTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=_server.serve_forever, daemon=True).start()
    url = f"http://127.0.0.1:{port}/index.html"
    print(f"Serving at {url}")
    webbrowser.open(url)


serve_html(render_response_to_html(response))

Serving at http://127.0.0.1:8765/index.html
